# XGBoost Phase 1: Recursive vs. Direct Multi-Step Comparison

This notebook implements the first phase of the XGBoost benchmark for ERCOT (Dallas) energy demand. We focus strictly on **lagging features** and compare forecasting strategies:

1. **Recursive Forecasting**: A single model trained to predict $t+1$.
2. **Direct Multi-Step (Seasonally Aligned)**: 24 independent models with aligned seasonal lags.
3. **Direct Multi-Step (Rich Lags)**: Adds "Origin Momentum" features ($y_{T-1}, y_{T-2}$) to the aligned model.

## Objective
Establish the strongest univariate XGBoost baseline using optimized lagging features.

In [1]:
import pandas as pd
import numpy as np
import glob
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# Constants
HORIZON = 24

## 1. Data Preparation

In [2]:
def load_ercot_data():
    files = sorted(glob.glob('../../data/ercot/raw/ercot_20*.xlsx'))
    dfs = []
    for f in files:
        df = pd.read_excel(f)
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    
    def parse_ercot_datetime(s):
        s = str(s).replace(' DST', '')
        if s.endswith('24:00'):
            return pd.to_datetime(s.replace('24:00', '00:00')) + pd.Timedelta(days=1) - pd.Timedelta(hours=1)
        else:
            return pd.to_datetime(s) - pd.Timedelta(hours=1)

    df['timestamp'] = df['Hour Ending'].apply(parse_ercot_datetime)
    df = df[['timestamp', 'NCENT']].copy()
    df.rename(columns={'NCENT': 'target'}, inplace=True)
    df = df.sort_values('timestamp').set_index('timestamp')
    df = df.resample('h').mean()
    df['target'] = df['target'].interpolate()
    return df

df = load_ercot_data()

## 2. Training Strategies

In [3]:
train_end = '2025-07-31 23:00:00'
train_df = df[:train_end].copy()

# --- Strategy A: Recursive ---
print("Training Strategy A (Recursive)...")
LAGS_REC = [1, 12, 24, 48, 168]
df_rec = df.copy()
for l in LAGS_REC:
    df_rec[f'lag_{l}'] = df_rec['target'].shift(l)

df_rec = df_rec.dropna()
train_rec = df_rec[:train_end]

model_rec = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
model_rec.fit(train_rec[[f'lag_{l}' for l in LAGS_REC]], train_rec['target'])

# --- Strategy B: Direct (Rich Lags) ---
print("Training Strategy B (Direct Rich Lags)...")
models_direct = {}
feature_cols = ['feat_origin', 'feat_origin_m1', 'feat_origin_m2', 
                'feat_24h_aligned', 'feat_48h_aligned', 'feat_168h_aligned']

for h in range(1, HORIZON + 1):
    df_h = df.copy()
    df_h['target_h'] = df_h['target'].shift(-h)
    
    # Origin + Momentum
    df_h['feat_origin'] = df_h['target']
    df_h['feat_origin_m1'] = df_h['target'].shift(1)
    df_h['feat_origin_m2'] = df_h['target'].shift(2)
    
    # Seasonally Aligned
    df_h['feat_24h_aligned'] = df_h['target'].shift(24 - h)
    df_h['feat_48h_aligned'] = df_h['target'].shift(48 - h)
    df_h['feat_168h_aligned'] = df_h['target'].shift(168 - h)
    
    df_h = df_h.dropna()
    train_h = df_h[:train_end]
    
    m = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    m.fit(train_h[feature_cols], train_h['target_h'])
    models_direct[h] = m

Training Strategy A (Recursive)...
Training Strategy B (Direct Rich Lags)...


## 3. Evaluation

In [4]:
def calculate_metrics(y_true, y_pred, y_train):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true)))
    mae_naive = mean_absolute_error(y_train[168:], y_train[:-168])
    mase = mae / mae_naive
    return {'MAE': mae, 'RMSE': rmse, 'sMAPE': smape, 'MASE': mase}

def evaluate_strategy(strategy, test_df):
    all_preds = []
    all_actuals = []
    origins = test_df[test_df.index.hour == 0].index
    for origin in origins:
        if origin + pd.Timedelta(hours=24) > test_df.index[-1]:
            continue
        actual = test_df.loc[origin + pd.Timedelta(hours=1) : origin + pd.Timedelta(hours=24), 'target'].values
        
        if strategy == 'recursive':
            preds = []
            for h in range(1, 25):
                curr_features = []
                for l in LAGS_REC:
                    target_time = origin + pd.Timedelta(hours=h) - pd.Timedelta(hours=l)
                    val = df.loc[target_time, 'target'] if target_time <= origin else preds[int((target_time-origin).total_seconds()//3600)-1]
                    curr_features.append(val)
                preds.append(model_rec.predict(np.array([curr_features]))[0])
            all_preds.append(preds)
        else:
            preds = []
            for h in range(1, 25):
                feats = pd.DataFrame([{
                    'feat_origin': df.loc[origin, 'target'],
                    'feat_origin_m1': df.loc[origin - pd.Timedelta(hours=1), 'target'],
                    'feat_origin_m2': df.loc[origin - pd.Timedelta(hours=2), 'target'],
                    'feat_24h_aligned': df.loc[origin - pd.Timedelta(hours=24-h), 'target'],
                    'feat_48h_aligned': df.loc[origin - pd.Timedelta(hours=48-h), 'target'],
                    'feat_168h_aligned': df.loc[origin - pd.Timedelta(hours=168-h), 'target']
                }])
                preds.append(models_direct[h].predict(feats)[0])
            all_preds.append(preds)
        all_actuals.append(actual)
    return np.array(all_actuals), np.array(all_preds)

aug_df = df['2025-08-01 00:00:00':'2025-08-31 23:00:00']
mar_df = df['2026-03-01 00:00:00':'2026-03-31 23:00:00']

for window_name, test_df in [("August 2025", aug_df), ("March 2026", mar_df)]:
    act, p_rec = evaluate_strategy('recursive', test_df)
    _, p_dir = evaluate_strategy('direct', test_df)
    print(f"\nResults for {window_name}:")
    print("Recursive:", calculate_metrics(act.flatten(), p_rec.flatten(), train_df['target']))
    print("Direct (Rich):", calculate_metrics(act.flatten(), p_dir.flatten(), train_df['target']))


Results for August 2025:
Recursive: {'MAE': 1796.7853724367185, 'RMSE': np.float64(2401.413237487391), 'sMAPE': np.float64(9.181385823762701), 'MASE': 0.9369475620341007}
Direct (Rich): {'MAE': 1180.2726875645833, 'RMSE': np.float64(1675.9575575270937), 'sMAPE': np.float64(5.991213313346182), 'MASE': 0.6154622773054769}

Results for March 2026:
Recursive: {'MAE': 1374.5132262744792, 'RMSE': np.float64(1832.074846903926), 'sMAPE': np.float64(10.113870904746525), 'MASE': 0.7167505012549051}
Direct (Rich): {'MAE': 959.1037529229167, 'RMSE': np.float64(1291.9779614977974), 'sMAPE': np.float64(6.954629243407096), 'MASE': 0.5001320340337602}
